In [4]:
# we need to run on a py file instead of a jupyter notebook otherwise multiprocessing will not work properly
import os
import sys

# agent.py / monte_carlo_tree_search.py / minimax.py import `Team2.data_processing`,
# so the repo root (the parent of Team2/) has to be on sys.path. This notebook lives in
# Team2/, so sys.path already covers `agent` and `model_files` themselves.
TEAM2_DIR = os.path.abspath("")
REPO_ROOT = os.path.dirname(TEAM2_DIR)
if REPO_ROOT not in sys.path:
    sys.path.insert(0, REPO_ROOT)

from agent import Agent, pit
from model_files.SLPolicyValueGPU import SLPolicyValueNetwork
import torch
import chess


device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("using device:", device)

# these weights live under backend/, not in Team2/model_weights/
WEIGHTS_PATH = os.path.join(
    REPO_ROOT, "Team2", "model_weights", "lab_trained_epoch_1.pth"
)

model1 = SLPolicyValueNetwork().to(device)
checkpoint = torch.load(WEIGHTS_PATH, map_location=device)
model1.load_state_dict(checkpoint["model"])
model1.eval()

agent = Agent(policy_value_network=model1, c_puct=1.0, dirichlet_alpha=0.3, dirichlet_epsilon=0.0)

mcts_policy_temp = 1.0 # setting temp to 1 means no temperature (temperature for policy)
mcts_temp = 1.0      # <1 means we boost the uct scores of top moves
sims = 500

using device: cpu


In [5]:
#human vs model

board = chess.Board()
human_turn = 1
stats = []
while not board.is_game_over():
    print(board, "\n")
    if human_turn == 1:
        move = input("enter a move in UCI format\n")
        if move == "q":
            break
        try:
            board.push_uci(move)
            human_turn *= -1
        except Exception:
            continue

    else:
        move, stats = agent.select_move(
            game_state=board,
            num_simulations=sims,
            temperature=0,
            debug=True,
            mcts_policy_temperature=mcts_policy_temp,
            mcts_temperature=mcts_temp,
        )
        print(f"model plays {board.san(chess.Move.from_uci(move))} ({move})")
        for rank, (uci, visits, ev, prior) in enumerate(stats[:5], start=1):
            print(f"{rank:>2}  {board.san(chess.Move.from_uci(uci)) + f' ({uci})':<16}{int(visits):>8}{ev:>+9.3f}{prior:>9.4f}")
        board.push_uci(move)
        human_turn *= -1
print(board, "\n")
print(board.result())


r n b q k b n r
p p p p p p p p
. . . . . . . .
. . . . . . . .
. . . . . . . .
. . . . . . . .
P P P P P P P P
R N B Q K B N R 

r n b q k b n r
p p p p p p p p
. . . . . . . .
. . . . . . . .
. . . P . . . .
. . . . . . . .
P P P . P P P P
R N B Q K B N R 

model plays Nf6 (g8f6)
 1  Nf6 (g8f6)            42   -0.072  12.4123
 2  d5 (d7d5)             41   -0.055  12.4135
 3  e6 (e7e6)             40   -0.068  11.0398
 4  d6 (d7d6)             36   -0.071  10.5452
 5  g6 (g7g6)             34   -0.104  10.3525
r n b q k b . r
p p p p p p p p
. . . . . n . .
. . . . . . . .
. . . P . . . .
. . . . . . . .
P P P . P P P P
R N B Q K B N R 

r n b q k b . r
p p p p p p p p
. . . . . n . .
. . . . . . . .
. . P P . . . .
. . . . . . . .
P P . . P P P P
R N B Q K B N R 

model plays g6 (g7g6)
 1  g6 (g7g6)             53   -0.078  12.7160
 2  e6 (e7e6)             47   -0.054  11.8157
 3  d6 (d7d6)             41   -0.083   9.6707
 4  c6 (c7c6)             36   -0.086   8.3861
 5  Nc6 (b8c

In [ ]:
board = chess.Board()
board.push_uci("e2e4")
board.push_uci("e7e5")
board.push_uci("f2f4")
stockfish_turn = 1
agent.stockfish.set_depth(15)
moves = []
move = None
while not board.is_game_over():
    print(board, "\n")
    if stockfish_turn == 1:
        agent.stockfish.set_fen_position(board.fen())
        print(agent.stockfish.get_evaluation()["value"] / 100)
        move = agent.stockfish.get_best_move()
        print(move)
        board.push_uci(move)
        stockfish_turn *= -1

    else:
        move, stats = agent.select_move(
            game_state=board,
            num_simulations=sims,
            temperature=0,
            debug=True,
            mcts_policy_temperature=mcts_policy_temp,
            mcts_temperature=mcts_temp,
        )
        board.push_uci(move)
        stockfish_turn *= -1
    moves.append(move)
    print(moves)


In [ ]:
agent.agent_vs_stockfish(
    2,
    sims,
    "pgn_files/demo_3200_sims_vs_depth16.pgn",
    mcts_policy_temperature=mcts_policy_temp,
    mcts_temperature=mcts_temp,
)
